# Zheng et al. (IEEE TCE 2025) — Baseline Simulation
### "Incentive-Driven Federated Learning for Collaborative Agricultural Consumer Electronics"
**Architecture Mapping:**
| Zheng et al. | Your System |
|---|---|
| ACE Edge Devices (Tasks m) | Cluster Heads (CHs) |
| Base Station (BS) | UAVs + Base Station |
| Local dataset D_m | CH aggregated data Q_h |
| Sub-channel c_mt | CH→UAV transmission resource |
| Local accuracy ε_mt | CH training accuracy |

**Algorithm:** Greedy Primal-Dual Auction (Algorithm 4, Zheng et al. TCE 2025)
- CHs bid with normalized utility Ω = (φ_mt − v_mt) / (θ·c_mt)
- BS selects winners greedily by descending Ω, subject to subchannel capacity C_max
- Winners transmit data to UAVs; UAVs do FL; evaluated on K = Σ G_u

**All physics parameters identical to proposed_fixed.ipynb (Singh et al. IEEE TSC 2024)**

## Cell 1 — Imports & Parameters

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.impute import SimpleImputer
import random, warnings
warnings.filterwarnings('ignore')

# ============================================================
# PHYSICS PARAMETERS — identical to proposed_fixed.ipynb
# ============================================================
AREA_SIZE    = 200
NUM_SENSORS  = 2000
K            = 70        # number of CHs
num_uavs     = 15

# Sensor/radio energy (Heinzelman 2002)
E_cir   = 50e-9
theta   = 10e-12
phi     = 0.0013e-12
D       = np.sqrt(theta / phi)  # crossover distance
B       = 20e6        # bandwidth Hz
p_h     = 0.2         # CH transmit power W
sigma   = 1e-9        # noise W
gamma   = 1
H_alt   = 100         # UAV altitude m
g_hu    = 1

# UAV propulsion (Zeng et al. IEEE TWC 2019)
C_PD    = 9.26e-4
C_RD    = 2250.0
V_u     = 10.0
v_amp   = 2.0
p_cir   = 0.1
E_MAX   = 10800.0     # UAV battery J

# FL computation/communication
mu_hu     = 1.0
W_model   = 1e6
f_hu      = 2e9
epsilon_u = 1e-26
alpha_u   = 0.5
beta_u    = 0.5
S_grad    = 1e6
b_u       = 20e6
SNR_u     = 20.0
P_com     = 0.1

# FL training
NUM_ROUNDS   = 5
LOCAL_EPOCHS = 3
LOCAL_BATCH  = 32
LR           = 1e-3
TEST_SIZE    = 0.2

# Reward (Singh et al. IEEE TSC 2024)
g_u_reward = 1.0
pi_u       = 1e-3

# ============================================================
# ZHENG et al. TCE 2025 — Greedy Nearest-UAV Assignment
# (No auction mechanism — uses same physics as proposed)
# ============================================================
# Channel params (reused from proposed physics)
H0_DB      = -80          # channel gain dB (for reference only)
N0_DBM     = -180         # noise PSD dBm/Hz (for reference only)

print("=" * 60)
print("  ZHENG et al. TCE 2025 — GREEDY NEAREST-UAV BASELINE")
print(f"  Farm: {AREA_SIZE}x{AREA_SIZE}m | CHs: {K} | UAVs: {num_uavs}")
print(f"  Assignment: Greedy nearest-UAV (no auction)")
print("=" * 60)

np.random.seed(42); random.seed(42)


## Cell 2 — Load Dataset

In [ ]:
try:
    df_full = pd.read_csv("agriculture_dataset.csv")
    print(f"Dataset loaded: {df_full.shape[0]:,} rows x {df_full.shape[1]} cols")
    USING_REAL = True
except FileNotFoundError:
    print("agriculture_dataset.csv not found — using synthetic data (3000 rows)")
    USING_REAL = False
    N = 3000
    df_full = pd.DataFrame({
        'High_Resolution_RGB'   : np.random.randint(0,2,N),
        'Multispectral_Images'  : np.random.randint(0,2,N),
        'Thermal_Images'        : np.random.randint(0,2,N),
        'Temporal_Images'       : np.random.randint(0,2,N),
        'Spatial_Resolution'    : np.random.uniform(0.01,2.7,N),
        'GPS_Coordinates'       : np.random.randint(100000,999999,N),
        'Field_Boundaries'      : np.random.randint(1,4,N),
        'Elevation_Data'        : np.random.uniform(10,100,N),
        'Canopy_Coverage'       : np.random.uniform(0,300,N),
        'NDVI'                  : np.random.uniform(0.05,1.05,N),
        'SAVI'                  : np.random.uniform(0.05,0.75,N),
        'Chlorophyll_Content'   : np.random.uniform(0.05,4.2,N),
        'Leaf_Area_Index'       : np.random.uniform(0.05,4.8,N),
        'Crop_Stress_Indicator' : np.random.randint(0,100,N),
        'Temperature'           : np.random.uniform(10,40,N),
        'Humidity'              : np.random.uniform(20,95,N),
        'Rainfall'              : np.random.uniform(0,130,N),
        'Wind_Speed'            : np.random.uniform(0.03,7.5,N),
        'Soil_Moisture'         : np.random.uniform(2,38,N),
        'Soil_pH'               : np.random.uniform(4.5,8.0,N),
        'Organic_Matter'        : np.random.uniform(0.001,20,N),
        'Pest_Hotspots'         : np.random.randint(0,2,N),
        'Weed_Coverage'         : np.random.uniform(0,9,N),
        'Pest_Damage'           : np.random.randint(0,100,N),
        'Crop_Growth_Stage'     : np.random.randint(1,5,N),
        'Expected_Yield'        : np.random.uniform(900,5200,N),
        'Crop_Type'             : np.random.choice(['Wheat','Maize','Rice','Soybean'],N),
        'Ground_Truth_Segmentation': np.random.randint(0,2,N),
        'Bounding_Boxes'        : np.random.randint(0,10,N),
        'Water_Flow'            : np.random.uniform(0,50,N),
        'Drainage_Features'     : np.random.randint(0,2,N),
    })
    lbl = np.zeros(N, dtype=int)
    for i in range(N):
        s = 0
        if 18 <= df_full.loc[i,'Temperature'] <= 34: s += 1
        if 5  <= df_full.loc[i,'Soil_pH'] <= 7.5:    s += 1
        if df_full.loc[i,'Soil_Moisture'] >= 15:      s += 1
        if df_full.loc[i,'NDVI'] >= 0.4:              s += 1
        if df_full.loc[i,'Crop_Stress_Indicator'] < 50: s += 1
        lbl[i] = 1 if s >= 3 else 0
    df_full['Crop_Health_Label'] = lbl

target_col   = 'Crop_Health_Label'
drop_cols    = ['Crop_Type','Ground_Truth_Segmentation','Bounding_Boxes']
feature_cols = [c for c in df_full.columns if c not in drop_cols + [target_col]]
df_full[feature_cols] = df_full[feature_cols].apply(pd.to_numeric, errors='coerce')
df_full[target_col]   = pd.to_numeric(df_full[target_col], errors='coerce')
imputer  = SimpleImputer(strategy='mean')
X_all_raw = imputer.fit_transform(df_full[feature_cols])
y_all    = df_full[target_col].fillna(0).values.astype(int)
scaler   = StandardScaler()
X_all    = scaler.fit_transform(X_all_raw)
N_FEATURES = X_all.shape[1]
print(f"Features: {N_FEATURES} | Samples: {len(y_all)} | "
      f"Healthy: {np.sum(y_all==1)} | Unhealthy: {np.sum(y_all==0)}")


## Cell 3 — Sensor Deployment + K-Means++ → Cluster Heads

In [ ]:
sensor_data = []
for i in range(NUM_SENSORS):
    x = np.random.uniform(0, AREA_SIZE)
    y = np.random.uniform(0, AREA_SIZE)
    sensor_data.append([f"SENS{str(i+1).zfill(4)}", x, y])

df = pd.DataFrame(sensor_data, columns=["sensor_id","x","y"])
df.set_index("sensor_id", inplace=True)
df["energy"] = np.random.uniform(2.0, 4.0, NUM_SENSORS)
df["k_n"]    = np.random.randint(2000, 6000, NUM_SENSORS)
df["I_np"]   = np.random.randint(0, 2, NUM_SENSORS)

kmeans = KMeans(n_clusters=K, init="k-means++", random_state=42, n_init=10)
df["cluster_id"] = kmeans.fit_predict(df[["x","y"]]).astype(int)

CH_list = []
for c in range(K):
    pts = df[df["cluster_id"]==c]
    cen = kmeans.cluster_centers_[c]
    dists = np.sqrt((pts["x"]-cen[0])**2 + (pts["y"]-cen[1])**2)
    CH_list.append(dists.idxmin())

stable_dict = {ch: [] for ch in CH_list}
for s_id, row in df.iterrows():
    stable_dict[CH_list[int(row["cluster_id"])]].append(s_id)

Q_h = {ch: sum(df.loc[s,"k_n"] for s in sensors)
       for ch, sensors in stable_dict.items()}

rho_h = {}
for h, ch_id in enumerate(CH_list):
    members = stable_dict[ch_id]
    rho_h[ch_id] = sum(df.loc[s,"I_np"] for s in members)

UAV_positions = [(np.random.uniform(0,AREA_SIZE),
                  np.random.uniform(0,AREA_SIZE)) for _ in range(num_uavs)]
UAV_BASE = np.array([AREA_SIZE/2, AREA_SIZE/2])

cluster_sizes = [len(stable_dict[ch]) for ch in CH_list]
print(f"Sensors: {NUM_SENSORS} | CHs: {K} | UAVs: {num_uavs}")
print(f"Avg cluster size: {np.mean(cluster_sizes):.1f}")

ch_table = pd.DataFrame({
    'CH'        : [f'CH{i+1:02d}' for i in range(K)],
    'Sensors'   : cluster_sizes,
    'Q_h(Mbits)': [round(Q_h[ch]/1e6, 2) for ch in CH_list],
    'rho_h'     : [round(rho_h[ch], 1) for ch in CH_list]
})
print("\n── CH Summary (first 10) ──")
print(ch_table.head(10).to_string(index=False))


## Cell 4 — Energy Functions (same physics as proposed)

In [ ]:
# ── Propulsion power (Zeng et al. TWC 2019) ──────────────────
def propulsion_power(v_speed):
    return C_PD * v_speed**3 + C_RD / v_speed

P_u = propulsion_power(V_u)   # UAV propulsion power W

# ── Eq.2: Sensor → CH transmission energy ────────────────────
def calc_E_sensor_tx(k_n, d):
    if d < D:
        return k_n * (theta * d**2 + E_cir)
    else:
        return k_n * (phi * d**4 + E_cir)

# ── Eq.rec: CH reception energy ──────────────────────────────
def calc_E_sensor_rx(C_nh, k_n):
    return C_nh * k_n * E_cir

# ── Data rate CH→UAV (5G link) ───────────────────────────────
def data_rate_R(ch_pos, uav_pos):
    dx = ch_pos[0] - uav_pos[0]
    dy = ch_pos[1] - uav_pos[1]
    d_2d = np.sqrt(dx**2 + dy**2)
    d_3d = np.sqrt(d_2d**2 + H_alt**2)
    h    = g_hu / (d_3d**2)
    snr  = (p_h * h) / sigma
    return B * np.log2(1 + snr)

# ── Eq.11: CH→UAV transmission energy ───────────────────────
def calc_E_CH(Q_bits, R_bps):
    T = Q_bits / max(R_bps, 1e-9)
    return p_h * T

# ── Eq.12: UAV traversal energy ─────────────────────────────
def calc_E_tve(route_length):
    return P_u * (route_length / V_u)

# ── Eq.16: UAV compute energy per CH ─────────────────────────
def calc_E_cmp_ch(Q_bits):
    return epsilon_u * (f_hu**2) * Q_bits * alpha_u * W_model

# ── Eq.17: Total UAV compute energy ──────────────────────────
def calc_E_cmp_total(matched_chs, Q_h_dict, CH_list_local):
    total = 0
    for h in matched_chs:
        total += calc_E_cmp_ch(Q_h_dict[CH_list_local[h]])
    return total

# ── Eq.20: UAV→BS communication energy ───────────────────────
def calc_E_com():
    snr_linear = 10 ** (SNR_u / 10)
    R_up = b_u * np.log2(1 + snr_linear)
    S_compressed = S_grad * alpha_u * beta_u
    T_com = S_compressed / R_up
    return T_com * P_com

# ── Transmission time ─────────────────────────────────────────
def trans_time_T(Q_bits, R_bps):
    return Q_bits / max(R_bps, 1e-9)

print(f"All energy functions ready.")
print(f"  P_u = {P_u:.1f} W  at v={V_u} m/s")
print(f"  D   = {D:.2f} m  (crossover distance)")


## Cell 5 — Zheng et al.: Greedy Nearest-UAV CH Assignment

**Assignment strategy (no auction):**
- All CHs participate (no bid filtering)
- Each CH is assigned to the nearest UAV that has remaining energy budget
- UAV traversal energy tracked; CHs that exceed UAV budget are unmatched
- Same energy equations (Eq.2, 11, 12, 17, 20) as proposed system
- Evaluated on same K_reward metric as proposed


In [ ]:
def zheng_greedy_assign(K_local, num_uavs_local, CH_list_local,
                          Q_h_local, UAV_pos_local, seed=42):
    """
    Zheng et al. baseline — Greedy Nearest-UAV CH→UAV assignment.
    No auction mechanism. All CHs compete; nearest energy-feasible UAV wins.

    Returns:
      M_h : list[int|None] — M_h[h] = UAV index assigned to CH h, or None
      M_u : list[list[int]] — M_u[u] = list of CH indices assigned to UAV u
      E_rem: list[float] — remaining energy per UAV after traversal
    """
    M_h   = [None] * K_local
    M_u   = [[] for _ in range(num_uavs_local)]
    E_rem = [E_MAX] * num_uavs_local

    for h in range(K_local):
        ch_id  = CH_list_local[h]
        ch_pos = (df.loc[ch_id, "x"], df.loc[ch_id, "y"])

        best_u    = None
        best_cost = np.inf
        for u in range(num_uavs_local):
            dx   = ch_pos[0] - UAV_pos_local[u][0]
            dy   = ch_pos[1] - UAV_pos_local[u][1]
            dist = np.sqrt(dx**2 + dy**2)
            e_tve = calc_E_tve(dist)
            if e_tve < E_rem[u] and e_tve < best_cost:
                best_cost = e_tve
                best_u    = u

        if best_u is not None:
            M_h[h]          = best_u
            M_u[best_u].append(h)
            E_rem[best_u]  -= best_cost

    matched_count = sum(1 for x in M_h if x is not None)
    print(f"Greedy assignment: {matched_count}/{K_local} CHs matched to UAVs")
    for u in range(num_uavs_local):
        if M_u[u]:
            print(f"  UAV{u+1:02d}: {len(M_u[u])} CHs | E_rem={E_rem[u]:.1f}J")

    return M_h, M_u, E_rem


# Run greedy assignment with full K=70, num_uavs=15
M_h, M_u, E_rem = zheng_greedy_assign(
    K, num_uavs, CH_list, Q_h, UAV_positions, seed=42)

print("\n── UAV Assignment Summary ──")
for u in range(num_uavs):
    if M_u[u]:
        print(f"  UAV{u+1:02d}: {len(M_u[u])} CHs | E_rem={E_rem[u]:.1f}J")


## Cell 6 — FL Training (FedAvg)

In [ ]:
from sklearn.linear_model import SGDClassifier

def local_train(X, y, global_coef, global_intercept, epochs=LOCAL_EPOCHS):
    if len(X) == 0 or len(np.unique(y)) < 2:
        return global_coef.copy(), global_intercept.copy()
    model = SGDClassifier(loss='log_loss', max_iter=epochs, learning_rate='constant',
                          eta0=LR, random_state=42)
    model.coef_      = global_coef.copy()
    model.intercept_ = global_intercept.copy()
    model.classes_   = np.array([0, 1])
    for _ in range(epochs):
        idx = np.random.permutation(len(X))
        for start in range(0, len(X), LOCAL_BATCH):
            batch = idx[start:start+LOCAL_BATCH]
            if len(batch) < 2: continue
            model.partial_fit(X[batch], y[batch], classes=[0,1])
    return model.coef_.copy(), model.intercept_.copy()

def fedavg(updates, weights):
    total_w = sum(weights)
    if total_w == 0:
        return updates[0][0], updates[0][1]
    agg_coef = sum(w * c for (c, _), w in zip(updates, weights)) / total_w
    agg_int  = sum(w * i for (_, i), w in zip(updates, weights)) / total_w
    return agg_coef, agg_int

# Data partitioning: assign rows to CHs
rows_per_cluster = max(1, len(df_full) // K)
shuffled_idx     = np.random.permutation(len(df_full))
cluster_row_map  = {}
for h in range(K):
    s = h * rows_per_cluster
    e = s + rows_per_cluster if h < K-1 else len(df_full)
    cluster_row_map[h] = shuffled_idx[s:e]

# Build per-UAV datasets from WINNER CHs only
uav_data   = {}
uav_travel = {}
for u in range(num_uavs):
    matched = M_u[u]
    if not matched:
        uav_data[u]   = (np.zeros((0, N_FEATURES)), np.zeros(0, dtype=int))
        uav_travel[u] = 0.0
        continue
    Xu, yu = [], []
    for h in matched:
        idx = cluster_row_map[h]
        Xu.append(X_all[idx]); yu.append(y_all[idx])
    uav_data[u] = (np.vstack(Xu), np.concatenate(yu))
    route = ([UAV_BASE] +
             [np.array([df.loc[CH_list[h],"x"], df.loc[CH_list[h],"y"]]) for h in matched] +
             [UAV_BASE])
    uav_travel[u] = sum(np.linalg.norm(route[i+1]-route[i]) for i in range(len(route)-1))

active_uavs = [u for u in range(num_uavs) if len(uav_data[u][1]) > 0]

# Global model init
g_coef = np.zeros((1, N_FEATURES))
g_int  = np.zeros(1)

# Held-out test set from full dataset
X_tr, X_te, y_tr, y_te = train_test_split(X_all, y_all, test_size=TEST_SIZE,
                                           random_state=42, stratify=y_all)

fl_acc_rounds = []
print(f"Active UAVs (have data): {len(active_uavs)}")
print("Starting FedAvg FL training...")

for rnd in range(NUM_ROUNDS):
    local_updates = []
    local_weights = []
    for u in active_uavs:
        X_u, y_u = uav_data[u]
        if len(X_u) < 2: continue
        c, i = local_train(X_u, y_u, g_coef, g_int, LOCAL_EPOCHS)
        local_updates.append((c, i))
        local_weights.append(len(y_u))
    if not local_updates: break
    g_coef, g_int = fedavg(local_updates, local_weights)

    # Evaluate
    from sklearn.linear_model import SGDClassifier as _S
    ev = _S(loss='log_loss'); ev.coef_ = g_coef; ev.intercept_ = g_int
    ev.classes_ = np.array([0,1])
    y_pred = ev.predict(X_te)
    acc    = accuracy_score(y_te, y_pred)
    fl_acc_rounds.append(acc)
    print(f"  Round {rnd+1}/{NUM_ROUNDS} — Accuracy: {acc:.4f}")

final_acc = fl_acc_rounds[-1] if fl_acc_rounds else 0.0
print(f"\nFinal FL Accuracy: {final_acc:.4f}")


## Cell 7 — Compute Energy Metrics & K_reward

In [ ]:
# ── Sensor→CH energies (same as proposed) ─────────────────────
E_sensor_tx = 0
for s_id, row in df.iterrows():
    ch_id = CH_list[int(row["cluster_id"])]
    d     = np.sqrt((row["x"] - df.loc[ch_id,"x"])**2 +
                    (row["y"] - df.loc[ch_id,"y"])**2)
    E_sensor_tx += calc_E_sensor_tx(row["k_n"], d)

E_sensor_rx = sum(
    calc_E_sensor_rx(1, df.loc[s,"k_n"])
    for ch in CH_list for s in stable_dict[ch])

# ── Eq.11: CH→UAV tx energy (winner CHs only) ─────────────────
E_ch_tx = 0
for h in range(K):
    if M_h[h] is not None:
        u      = M_h[h]
        ch_pos = (df.loc[CH_list[h],"x"], df.loc[CH_list[h],"y"])
        R      = data_rate_R(ch_pos, UAV_positions[u])
        E_ch_tx += calc_E_CH(Q_h[CH_list[h]], R)

# ── Eq.12: UAV traversal energy ──────────────────────────────
E_tve = sum(E_MAX - E_rem[u] for u in range(num_uavs))

# ── Eq.17: UAV computation energy ─────────────────────────────
E_cmp = sum(calc_E_cmp_total(M_u[u], Q_h, CH_list) for u in range(num_uavs))

# ── Eq.20: UAV→BS communication energy ───────────────────────
E_com = num_uavs * calc_E_com()

# ── Transmission time ─────────────────────────────────────────
T_trans = sum(
    trans_time_T(Q_h[CH_list[h]],
                 data_rate_R((df.loc[CH_list[h],"x"], df.loc[CH_list[h],"y"]),
                             UAV_positions[M_h[h]]))
    for h in range(K) if M_h[h] is not None)

E_total = E_sensor_tx + E_sensor_rx + E_ch_tx + E_tve + E_cmp + E_com

# ── K_reward = Σ G_u  (Eq.22-25, Singh et al. 2024) ──────────
# G_u = g_u * R_u - π_u * (E_tare + E_cmp + E_com)
# R_u = acc (FL accuracy reward)
matched_count = sum(1 for x in M_h if x is not None)

K_reward = 0.0
for u in range(num_uavs):
    if not M_u[u]:
        continue
    E_tare_u  = E_MAX - E_rem[u]   # traversal energy this UAV used
    E_cmp_u   = calc_E_cmp_total(M_u[u], Q_h, CH_list)
    E_com_u   = calc_E_com()
    R_u       = g_u_reward * final_acc
    G_u       = R_u - pi_u * (E_tare_u + E_cmp_u + E_com_u)
    K_reward += G_u

total_cost = pi_u * E_total

print("── ZHENG et al. TCE 2025 — Main Simulation Energy ──")
print(f"  E_sensor_tx (Eq.2)  : {E_sensor_tx:.4e} J")
print(f"  E_sensor_rx (Eq.rec): {E_sensor_rx:.4e} J")
print(f"  E_CH_tx     (Eq.11) : {E_ch_tx:.4e} J")
print(f"  E_UAV_tve   (Eq.12) : {E_tve:.4e} J")
print(f"  E_UAV_cmp   (Eq.17) : {E_cmp:.4e} J")
print(f"  E_UAV_com   (Eq.20) : {E_com:.4e} J")
print(f"  E_total             : {E_total:.4e} J")
print(f"  T_trans             : {T_trans:.4e} s")
print(f"  Matched CHs         : {matched_count}/{K}")
print(f"  FL Accuracy         : {final_acc:.4f}")
print(f"  Total Cost          : {total_cost:.4e}")
print(f"  K_reward (Eq.25)    : {K_reward:.6f}")


## Cell 8 — Parametric Sweep: run_one function

In [ ]:
def run_one_zheng(n_sensors, n_uavs, K_ch=None, seed=42):
    """
    Run one simulation point with Zheng et al. greedy nearest-UAV assignment.
    No auction mechanism. All CHs assigned to nearest energy-feasible UAV.
    Same physics as proposed (Eq.2, 11, 12, 17, 20).
    """
    rng = np.random.RandomState(seed)

    if K_ch is None:
        K_ch = max(5, n_sensors // 30)

    # ── Sensor deployment ──────────────────────────────────────
    sx = rng.uniform(0, AREA_SIZE, n_sensors)
    sy = rng.uniform(0, AREA_SIZE, n_sensors)
    k_n = rng.randint(2000, 6000, n_sensors)

    # ── K-Means CH selection ───────────────────────────────────
    coords = np.column_stack([sx, sy])
    km = KMeans(n_clusters=K_ch, init='k-means++', random_state=seed, n_init=10)
    labels = km.fit_predict(coords)

    CH_pos_x = np.zeros(K_ch); CH_pos_y = np.zeros(K_ch)
    Q_h_arr  = np.zeros(K_ch)
    for c in range(K_ch):
        mask = (labels == c)
        cen  = km.cluster_centers_[c]
        dists = np.sqrt((sx[mask]-cen[0])**2 + (sy[mask]-cen[1])**2)
        idx_ch = np.where(mask)[0][np.argmin(dists)]
        CH_pos_x[c] = sx[idx_ch]; CH_pos_y[c] = sy[idx_ch]
        Q_h_arr[c]  = np.sum(k_n[mask])

    # ── UAV positions ──────────────────────────────────────────
    uav_px = rng.uniform(0, AREA_SIZE, n_uavs)
    uav_py = rng.uniform(0, AREA_SIZE, n_uavs)

    # ── Greedy Nearest-UAV Assignment (no auction) ─────────────
    M_h_l   = [None] * K_ch
    M_u_l   = [[] for _ in range(n_uavs)]
    E_rem_l = [E_MAX] * n_uavs

    for h in range(K_ch):
        best_u = None; best_cost = np.inf
        for u in range(n_uavs):
            dist  = np.sqrt((CH_pos_x[h]-uav_px[u])**2 + (CH_pos_y[h]-uav_py[u])**2)
            e_tve = calc_E_tve(dist)
            if e_tve < E_rem_l[u] and e_tve < best_cost:
                best_cost = e_tve; best_u = u
        if best_u is not None:
            M_h_l[h] = best_u
            M_u_l[best_u].append(h)
            E_rem_l[best_u] -= best_cost

    matched_count = sum(1 for x in M_h_l if x is not None)

    # ── Sensor→CH energies (Eq.2, Eq.rec) ─────────────────────
    E_tx_total = 0; E_rx_total = 0
    for i in range(n_sensors):
        c = labels[i]
        d = np.sqrt((sx[i]-CH_pos_x[c])**2 + (sy[i]-CH_pos_y[c])**2)
        E_tx_total += calc_E_sensor_tx(k_n[i], d)
        E_rx_total += calc_E_sensor_rx(1, k_n[i])

    # ── CH→UAV tx energy (Eq.11) ──────────────────────────────
    E_ch_tx_l = 0
    for h in range(K_ch):
        if M_h_l[h] is None: continue
        u    = M_h_l[h]
        dx   = CH_pos_x[h] - uav_px[u]; dy = CH_pos_y[h] - uav_py[u]
        d_3d = np.sqrt(dx**2 + dy**2 + H_alt**2)
        h_ch = g_hu / (d_3d**2)
        snr  = (p_h * h_ch) / sigma
        R    = B * np.log2(1 + snr)
        T    = Q_h_arr[h] / max(R, 1e-9)
        E_ch_tx_l += p_h * T

    # ── UAV traversal energy (Eq.12) ──────────────────────────
    E_tve_l = sum(E_MAX - E_rem_l[u] for u in range(n_uavs))

    # ── UAV computation energy (Eq.17) ────────────────────────
    E_cmp_l = 0
    for u in range(n_uavs):
        for h in M_u_l[u]:
            E_cmp_l += epsilon_u * (f_hu**2) * Q_h_arr[h] * alpha_u * W_model

    # ── UAV→BS communication energy (Eq.20) ───────────────────
    snr_lin = 10 ** (SNR_u / 10)
    R_up    = b_u * np.log2(1 + snr_lin)
    T_com   = (S_grad * alpha_u * beta_u) / R_up
    E_com_l = n_uavs * T_com * P_com

    # ── Transmission time ──────────────────────────────────────
    T_trans_l = 0
    for h in range(K_ch):
        if M_h_l[h] is None: continue
        u    = M_h_l[h]
        dx   = CH_pos_x[h] - uav_px[u]; dy = CH_pos_y[h] - uav_py[u]
        d_3d = np.sqrt(dx**2 + dy**2 + H_alt**2)
        h_ch = g_hu / (d_3d**2)
        snr  = (p_h * h_ch) / sigma
        R    = B * np.log2(1 + snr)
        T_trans_l += Q_h_arr[h] / max(R, 1e-9)

    E_total_l = E_tx_total + E_rx_total + E_ch_tx_l + E_tve_l + E_cmp_l + E_com_l

    # ── FL simulation ─────────────────────────────────────────
    coverage  = matched_count / max(K_ch, 1)
    n_samples = max(50, int(len(X_all) * coverage))
    idx_s = rng.choice(len(X_all), size=min(n_samples, len(X_all)), replace=False)
    X_sub = X_all[idx_s]; y_sub = y_all[idx_s]
    if len(np.unique(y_sub)) >= 2:
        X_tr2, X_te2, y_tr2, y_te2 = train_test_split(
            X_sub, y_sub, test_size=TEST_SIZE, random_state=seed)
        model_fl = SGDClassifier(loss='log_loss', max_iter=NUM_ROUNDS*LOCAL_EPOCHS,
                                 learning_rate='constant', eta0=LR, random_state=seed)
        if len(np.unique(y_tr2)) >= 2:
            model_fl.fit(X_tr2, y_tr2)
            acc = accuracy_score(y_te2, model_fl.predict(X_te2))
        else:
            acc = float(np.mean(y_te2 == y_tr2[0]))
    else:
        acc = 0.5

    # ── K_reward (Eq.22-25) ───────────────────────────────────
    K_reward_l = 0.0
    for u in range(n_uavs):
        if not M_u_l[u]: continue
        E_tare_u = E_MAX - E_rem_l[u]
        E_cmp_u  = sum(epsilon_u * (f_hu**2) * Q_h_arr[h] * alpha_u * W_model
                       for h in M_u_l[u])
        E_com_u  = T_com * P_com
        R_u      = g_u_reward * acc
        G_u      = R_u - pi_u * (E_tare_u + E_cmp_u + E_com_u)
        K_reward_l += G_u

    return {
        'n_sensors'  : n_sensors,
        'K'          : K_ch,
        'n_uavs'     : n_uavs,
        'matched'    : matched_count,
        'E_tx'       : E_tx_total,
        'E_rx'       : E_rx_total,
        'T_trans'    : T_trans_l,
        'E_total'    : E_total_l,
        'total_cost' : pi_u * E_total_l,
        'K_reward'   : K_reward_l,
        'acc'        : acc,
    }

print("run_one_zheng() defined — Greedy Nearest-UAV (no auction).")


## Cell 9 — Case 1: Varying Sensors (UAVs=7)

In [ ]:
from sklearn.linear_model import SGDClassifier

sensor_range_1 = [200, 400, 600, 800, 1000]
results1 = []
for ns in sensor_range_1:
    r = run_one_zheng(ns, 7, seed=42)
    results1.append(r)
    print(f"  n_sensors={ns:5d} | K={r['K']:2d} | matched={r['matched']:2d} | "
          f"E_total={r['E_total']:.2e} | K_reward={r['K_reward']:.4f} | acc={r['acc']:.4f}")

df1 = pd.DataFrame(results1)
print("\nCase 1 complete.")
print(df1[['n_sensors','K','n_uavs','matched','E_tx','E_rx',
           'T_trans','E_total','total_cost','K_reward','acc']].to_string(index=False))


## Cell 10 — Case 2: Varying Sensors (UAVs=15)

In [ ]:
sensor_range_2 = list(range(200, 2001, 200))
results2 = []
for ns in sensor_range_2:
    r = run_one_zheng(ns, 15, seed=42)
    results2.append(r)
    print(f"  n_sensors={ns:5d} | K={r['K']:2d} | matched={r['matched']:2d} | "
          f"E_total={r['E_total']:.2e} | K_reward={r['K_reward']:.4f} | acc={r['acc']:.4f}")

df2 = pd.DataFrame(results2)
print("\nCase 2 complete.")
print(df2[['n_sensors','K','n_uavs','matched','E_tx','E_rx',
           'T_trans','E_total','total_cost','K_reward','acc']].to_string(index=False))


## Cell 11 — Plots (matching proposed_fixed.ipynb style)
### Topology plots + 12 metric plots (bar & line)


In [ ]:
# ============================================================
# ZHENG et al. TCE 2025 — ALL PLOTS (matching proposed style)
# ============================================================
from scipy.spatial import Voronoi, voronoi_plot_2d

COLORS_CH  = plt.cm.tab20(np.linspace(0, 1, K))
COLORS_UAV = plt.cm.Set1(np.linspace(0, 0.85, num_uavs))
ALGO_LABEL  = 'Zheng et al. (TCE 2025)'

# ── Plot 1 — K-Means++ Clustering with Voronoi Boundaries ───────
fig, ax = plt.subplots(figsize=(9, 9))
try:
    vor = Voronoi(kmeans.cluster_centers_)
    voronoi_plot_2d(vor, ax=ax, show_vertices=False, line_colors='gray',
                    line_width=0.8, line_alpha=0.5, point_size=0)
except Exception:
    pass
sub = df.sample(min(500, len(df)), random_state=1)
ax.scatter(sub["x"], sub["y"], c=sub["cluster_id"], cmap="tab20",
           s=8, alpha=0.35, label="Sensors")
for h, ch_id in enumerate(CH_list):
    ax.scatter(df.loc[ch_id, "x"], df.loc[ch_id, "y"],
               color=COLORS_CH[h], s=100, marker='D',
               edgecolors='k', lw=0.8, zorder=4)
ax.plot([0, AREA_SIZE, AREA_SIZE, 0, 0],
        [0, 0, AREA_SIZE, AREA_SIZE, 0], 'k-', lw=2)
ax.set_title(f'K-Means++ Clustering with Voronoi Boundaries\n'
             f'Sensors={NUM_SENSORS}, CHs={K}',
             fontweight='bold', fontsize=12)
ax.set_xlabel('X (m)', fontsize=11); ax.set_ylabel('Y (m)', fontsize=11)
ax.legend(fontsize=9); ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig('zheng_plot01_kmeans_voronoi.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved: zheng_plot01_kmeans_voronoi.png')


# ── Plot 2 — CH → UAV Greedy Assignment ──────────────────────────
fig, ax = plt.subplots(figsize=(9, 9))
sub = df.sample(min(400, len(df)), random_state=1)
ax.scatter(sub["x"], sub["y"], c=sub["cluster_id"], cmap="tab20",
           s=6, alpha=0.3, label="Sensors")
for h, ch_id in enumerate(CH_list):
    ax.scatter(df.loc[ch_id, "x"], df.loc[ch_id, "y"],
               color=COLORS_CH[h], s=80, marker='D',
               edgecolors='k', lw=0.7, zorder=4)
for u in range(num_uavs):
    ux, uy = UAV_positions[u]
    ax.scatter(ux, uy, color=COLORS_UAV[u], s=180, marker='^',
               edgecolors='k', lw=1.2, zorder=5)
    ax.annotate(f'U{u+1}', (ux, uy), xytext=(3, 3),
                textcoords='offset points', fontsize=7,
                color=COLORS_UAV[u], fontweight='bold')
    for h in M_u[u]:
        cx = df.loc[CH_list[h], "x"]; cy = df.loc[CH_list[h], "y"]
        ax.plot([ux, cx], [uy, cy], color=COLORS_UAV[u],
                lw=1.8, ls='--', alpha=0.7, zorder=3)
ax.scatter(*UAV_BASE, color='red', s=300, marker='*',
           edgecolors='k', lw=1.5, zorder=6, label='Base')
ax.plot([0, AREA_SIZE, AREA_SIZE, 0, 0],
        [0, 0, AREA_SIZE, AREA_SIZE, 0], 'k-', lw=2)
matched_count = sum(1 for x in M_h if x is not None)
ax.set_title(f'CH → UAV Greedy Nearest Assignment\nMatched: {matched_count}/{K} CHs',
             fontweight='bold', fontsize=12)
ax.set_xlabel('X (m)', fontsize=11); ax.set_ylabel('Y (m)', fontsize=11)
ax.legend(fontsize=9); ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig('zheng_plot02_ch_uav_matching.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved: zheng_plot02_ch_uav_matching.png')


# ── Plot 3 — Sensor → CH Cluster Assignment ───────────────────────
fig, ax = plt.subplots(figsize=(9, 9))
ax.scatter(df["x"], df["y"], c=df["cluster_id"], cmap="tab20",
           s=8, alpha=0.4, label="Sensors")
for h, ch_id in enumerate(CH_list):
    hx, hy = df.loc[ch_id, "x"], df.loc[ch_id, "y"]
    ax.scatter(hx, hy, color=COLORS_CH[h], s=120, marker='*',
               edgecolors='k', lw=0.8, zorder=4)
    for s_id in stable_dict[ch_id]:
        sx, sy = df.loc[s_id, "x"], df.loc[s_id, "y"]
        ax.plot([sx, hx], [sy, hy], lw=2.5, alpha=0.75, color=COLORS_CH[h])
ax.plot([0, AREA_SIZE, AREA_SIZE, 0, 0],
        [0, 0, AREA_SIZE, AREA_SIZE, 0], 'k-', lw=2)
ax.set_title('Sensor → CH Cluster Assignment (K-Means++)',
             fontweight='bold', fontsize=12)
ax.set_xlabel('X (m)', fontsize=11); ax.set_ylabel('Y (m)', fontsize=11)
ax.legend(fontsize=9); ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig('zheng_plot03_sensor_ch_matching.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved: zheng_plot03_sensor_ch_matching.png')


# ── Plot 4 — UAV Load (CHs per UAV) ──────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
uav_load = [len(M_u[u]) for u in range(num_uavs)]
bars = ax.bar([f'UAV{u+1}' for u in range(num_uavs)], uav_load,
              color=[COLORS_UAV[u] for u in range(num_uavs)],
              edgecolor='k', lw=0.7, alpha=0.85)
for bar, v in zip(bars, uav_load):
    ax.text(bar.get_x()+bar.get_width()/2, v+0.05, str(v),
            ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_title('UAV Load: CHs Assigned per UAV (Greedy Nearest)',
             fontweight='bold', fontsize=12)
ax.set_ylabel('Number of CHs', fontsize=11)
ax.tick_params(axis='x', labelsize=9, rotation=45)
ax.grid(True, alpha=0.2, axis='y')
plt.tight_layout()
plt.savefig('zheng_plot04_uav_load.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved: zheng_plot04_uav_load.png')


# ── Plot 5 — Transmission Energy E_tx (independent of UAV count) ─
fig, ax = plt.subplots(figsize=(10, 6))
x_vals = df2['n_sensors'].tolist()
y_vals = df2['E_tx'].tolist()
bars = ax.bar(range(len(x_vals)), y_vals, color='steelblue',
              edgecolor='black', linewidth=0.8, alpha=0.88, width=0.55)
for bar, v in zip(bars, y_vals):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height(),
            f'{v:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_title('Transmission Energy E_tx (independent of UAV count)',
             fontweight='bold', fontsize=13)
ax.set_xlabel('Number of Sensors', fontsize=11)
ax.set_ylabel('E_tx (J)', fontsize=11)
ax.set_xticks(range(len(x_vals)))
ax.set_xticklabels([str(x) for x in x_vals], fontsize=9)
ax.grid(True, alpha=0.3, axis='y')
ax.text(0.98, 0.02, 'UAV count has no effect on this metric',
        transform=ax.transAxes, fontsize=8, ha='right', va='bottom', color='gray')
ax.text(0.98, 0.06, 'Sensor→CH energy (E_cir + path loss)',
        transform=ax.transAxes, fontsize=8, ha='right', va='bottom', color='gray')
plt.tight_layout()
plt.savefig('zheng_plot05_Etx.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved: zheng_plot05_Etx.png')


# ── Plot 6 — Reception Energy E_rx (independent of UAV count) ────
fig, ax = plt.subplots(figsize=(10, 6))
x_vals = df2['n_sensors'].tolist()
y_vals = df2['E_rx'].tolist()
bars = ax.bar(range(len(x_vals)), y_vals, color='darkorange',
              edgecolor='black', linewidth=0.8, alpha=0.88, width=0.55)
for bar, v in zip(bars, y_vals):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height(),
            f'{v:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_title('Reception Energy E_rx (independent of UAV count)',
             fontweight='bold', fontsize=13)
ax.set_xlabel('Number of Sensors', fontsize=11)
ax.set_ylabel('E_rx (J)', fontsize=11)
ax.set_xticks(range(len(x_vals)))
ax.set_xticklabels([str(x) for x in x_vals], fontsize=9)
ax.grid(True, alpha=0.3, axis='y')
ax.text(0.98, 0.02, 'UAV count has no effect on this metric',
        transform=ax.transAxes, fontsize=8, ha='right', va='bottom', color='gray')
ax.text(0.98, 0.06, 'CH receive energy (k_n × E_cir)',
        transform=ax.transAxes, fontsize=8, ha='right', va='bottom', color='gray')
plt.tight_layout()
plt.savefig('zheng_plot06_Erx.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved: zheng_plot06_Erx.png')


# ── Plot 7 — Total System Energy E_total — Case 1 (UAVs=7) ──────
fig, ax = plt.subplots(figsize=(10, 6))
x_vals = df1['n_sensors'].tolist()
y_vals = df1['E_total'].tolist()
bars = ax.bar(range(len(x_vals)), y_vals, color='crimson',
              edgecolor='black', linewidth=0.8, alpha=0.88, width=0.55)
for bar, v in zip(bars, y_vals):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height(),
            f'{v:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_title('Total System Energy E_total — Case 1 (UAVs=7)',
             fontweight='bold', fontsize=13)
ax.set_xlabel('Number of Sensors  [Case 1: UAVs=7, S=200→1000]', fontsize=11)
ax.set_ylabel('E_total (J)', fontsize=11)
ax.set_xticks(range(len(x_vals)))
ax.set_xticklabels([str(x) for x in x_vals], fontsize=9)
ax.grid(True, alpha=0.3, axis='y')
ax.text(0.98, 0.02, 'E_tx+E_rx+E_CH→UAV+E_tve',
        transform=ax.transAxes, fontsize=8, ha='right', va='bottom', color='gray')
plt.tight_layout()
plt.savefig('zheng_plot07_Etotal_case1.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved: zheng_plot07_Etotal_case1.png')


# ── Plot 8 — Total System Energy E_total — Case 2 (UAVs=15) ─────
fig, ax = plt.subplots(figsize=(10, 6))
x_vals = df2['n_sensors'].tolist()
y_vals = df2['E_total'].tolist()
bars = ax.bar(range(len(x_vals)), y_vals, color='firebrick',
              edgecolor='black', linewidth=0.8, alpha=0.88, width=0.55)
for bar, v in zip(bars, y_vals):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height(),
            f'{v:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_title('Total System Energy E_total — Case 2 (UAVs=15)',
             fontweight='bold', fontsize=13)
ax.set_xlabel('Number of Sensors  [Case 2: UAVs=15, S=200→2000]', fontsize=11)
ax.set_ylabel('E_total (J)', fontsize=11)
ax.set_xticks(range(len(x_vals)))
ax.set_xticklabels([str(x) for x in x_vals], fontsize=9)
ax.grid(True, alpha=0.3, axis='y')
ax.text(0.98, 0.02, 'E_tx+E_rx+E_CH→UAV+E_tve',
        transform=ax.transAxes, fontsize=8, ha='right', va='bottom', color='gray')
plt.tight_layout()
plt.savefig('zheng_plot08_Etotal_case2.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved: zheng_plot08_Etotal_case2.png')


# ── Plot 9 — Transmission Time T_trans — Case 1 (UAVs=7) ────────
fig, ax = plt.subplots(figsize=(10, 6))
x_vals = df1['K'].tolist()
y_vals = df1['T_trans'].tolist()
bars = ax.bar(range(len(x_vals)), y_vals, color='purple',
              edgecolor='black', linewidth=0.8, alpha=0.88, width=0.55)
for bar, v in zip(bars, y_vals):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height(),
            f'{v:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_title('Total Transmission Time T_trans — Case 1 (UAVs=7)',
             fontweight='bold', fontsize=13)
ax.set_xlabel('Number of Cluster Heads K (dynamic)', fontsize=11)
ax.set_ylabel('T_trans (s)', fontsize=11)
ax.set_xticks(range(len(x_vals)))
ax.set_xticklabels([str(k) for k in x_vals], fontsize=9)
ax.grid(True, alpha=0.3, axis='y')
ax.text(0.98, 0.02, 'More CHs → more total Q_h → higher T_trans',
        transform=ax.transAxes, fontsize=8, ha='right', va='bottom', color='gray')
sensors = df1['n_sensors'].tolist()
for i, (k, s) in enumerate(zip(x_vals, sensors)):
    ax.text(i, -ax.get_ylim()[1]*0.06, f'N={s}',
            ha='center', fontsize=7, color='dimgray')
ax.text(0.02, -0.08, 'Sensors:', transform=ax.transAxes, fontsize=7, color='dimgray')
plt.tight_layout()
plt.savefig('zheng_plot09_Ttrans_case1.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved: zheng_plot09_Ttrans_case1.png')


# ── Plot 10 — Transmission Time T_trans — Case 2 (UAVs=15) ──────
fig, ax = plt.subplots(figsize=(10, 6))
x_vals = df2['K'].tolist()
y_vals = df2['T_trans'].tolist()
bars = ax.bar(range(len(x_vals)), y_vals, color='mediumvioletred',
              edgecolor='black', linewidth=0.8, alpha=0.88, width=0.55)
for bar, v in zip(bars, y_vals):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height(),
            f'{v:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_title('Total Transmission Time T_trans — Case 2 (UAVs=15)',
             fontweight='bold', fontsize=13)
ax.set_xlabel('Number of Cluster Heads K (dynamic)', fontsize=11)
ax.set_ylabel('T_trans (s)', fontsize=11)
ax.set_xticks(range(len(x_vals)))
ax.set_xticklabels([str(k) for k in x_vals], fontsize=9)
ax.grid(True, alpha=0.3, axis='y')
ax.text(0.98, 0.02, 'More CHs → more total Q_h → higher T_trans',
        transform=ax.transAxes, fontsize=8, ha='right', va='bottom', color='gray')
sensors = df2['n_sensors'].tolist()
for i, (k, s) in enumerate(zip(x_vals, sensors)):
    ax.text(i, -ax.get_ylim()[1]*0.06, f'N={s}',
            ha='center', fontsize=7, color='dimgray')
ax.text(0.02, -0.08, 'Sensors:', transform=ax.transAxes, fontsize=7, color='dimgray')
plt.tight_layout()
plt.savefig('zheng_plot10_Ttrans_case2.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved: zheng_plot10_Ttrans_case2.png')


# ── Plot 11 — Total Cost — Case 1 (UAVs=7) ──────────────────────
fig, ax = plt.subplots(figsize=(10, 6))
x_vals = df1['n_sensors'].tolist()
y_vals = df1['total_cost'].tolist()
ax.plot(x_vals, y_vals, marker='o', lw=3, ms=10,
        color='saddlebrown', label='Total Cost')
ax.fill_between(x_vals, y_vals, alpha=0.12, color='saddlebrown')
for x, y in zip(x_vals, y_vals):
    ax.annotate(f'{y:.4f}', (x, y), textcoords='offset points',
                xytext=(0, 10), ha='center', fontsize=9, fontweight='bold')
ax.set_title('Total Cost — Case 1 (UAVs=7)', fontweight='bold', fontsize=13)
ax.set_xlabel('Number of Sensors  [Case 1: UAVs=7, N=200→1000]', fontsize=11)
ax.set_ylabel('Total Cost', fontsize=11)
ax.set_xticks(x_vals)
ax.tick_params(axis='x', labelsize=9)
ax.legend(fontsize=10); ax.grid(True, alpha=0.3)
ax.text(0.98, 0.02, 'π_u × E_total',
        transform=ax.transAxes, fontsize=8, ha='right', va='bottom', color='gray')
plt.tight_layout()
plt.savefig('zheng_plot11_cost_case1.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved: zheng_plot11_cost_case1.png')


# ── Plot 12 — Total Cost — Case 2 (UAVs=15) ─────────────────────
fig, ax = plt.subplots(figsize=(10, 6))
x_vals = df2['n_sensors'].tolist()
y_vals = df2['total_cost'].tolist()
ax.plot(x_vals, y_vals, marker='o', lw=3, ms=10,
        color='chocolate', label='Total Cost')
ax.fill_between(x_vals, y_vals, alpha=0.12, color='chocolate')
for x, y in zip(x_vals, y_vals):
    ax.annotate(f'{y:.4f}', (x, y), textcoords='offset points',
                xytext=(0, 10), ha='center', fontsize=9, fontweight='bold')
ax.set_title('Total Cost — Case 2 (UAVs=15)', fontweight='bold', fontsize=13)
ax.set_xlabel('Number of Sensors  [Case 2: UAVs=15, N=200→2000]', fontsize=11)
ax.set_ylabel('Total Cost', fontsize=11)
ax.set_xticks(x_vals)
ax.tick_params(axis='x', labelsize=9)
ax.legend(fontsize=10); ax.grid(True, alpha=0.3)
ax.text(0.98, 0.02, 'π_u × E_total',
        transform=ax.transAxes, fontsize=8, ha='right', va='bottom', color='gray')
plt.tight_layout()
plt.savefig('zheng_plot12_cost_case2.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved: zheng_plot12_cost_case2.png')


# ── Plot 13 — Total Revenue K (Eq.25) — Case 1 (UAVs=7) ────────
fig, ax = plt.subplots(figsize=(10, 6))
x_vals = df1['n_sensors'].tolist()
y_vals = df1['K_reward'].tolist()
ax.plot(x_vals, y_vals, marker='o', lw=3, ms=10,
        color='darkgreen', label='K = Σ G_u')
ax.fill_between(x_vals, y_vals, alpha=0.12, color='darkgreen')
for x, y in zip(x_vals, y_vals):
    ax.annotate(f'{y:.4f}', (x, y), textcoords='offset points',
                xytext=(0, 10), ha='center', fontsize=9, fontweight='bold')
ax.set_title('Total Revenue K (Eq.25) — Case 1 (UAVs=7)',
             fontweight='bold', fontsize=13)
ax.set_xlabel('Number of Sensors  [Case 1: UAVs=7, N=200→1000]', fontsize=11)
ax.set_ylabel('K = Σ G_u', fontsize=11)
ax.set_xticks(x_vals)
ax.tick_params(axis='x', labelsize=9)
ax.legend(fontsize=10); ax.grid(True, alpha=0.3)
ax.text(0.98, 0.02, 'G_u = R_u − π_u(E_tare+E_cmp+E_com)',
        transform=ax.transAxes, fontsize=8, ha='right', va='bottom', color='gray')
plt.tight_layout()
plt.savefig('zheng_plot13_reward_case1.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved: zheng_plot13_reward_case1.png')


# ── Plot 14 — Total Revenue K (Eq.25) — Case 2 (UAVs=15) ────────
fig, ax = plt.subplots(figsize=(10, 6))
x_vals = df2['n_sensors'].tolist()
y_vals = df2['K_reward'].tolist()
ax.plot(x_vals, y_vals, marker='o', lw=3, ms=10,
        color='seagreen', label='K = Σ G_u')
ax.fill_between(x_vals, y_vals, alpha=0.12, color='seagreen')
for x, y in zip(x_vals, y_vals):
    ax.annotate(f'{y:.4f}', (x, y), textcoords='offset points',
                xytext=(0, 10), ha='center', fontsize=9, fontweight='bold')
ax.set_title('Total Revenue K (Eq.25) — Case 2 (UAVs=15)',
             fontweight='bold', fontsize=13)
ax.set_xlabel('Number of Sensors  [Case 2: UAVs=15, S=200→2000]', fontsize=11)
ax.set_ylabel('K = Σ G_u', fontsize=11)
ax.set_xticks(x_vals)
ax.tick_params(axis='x', labelsize=9)
ax.legend(fontsize=10); ax.grid(True, alpha=0.3)
ax.text(0.98, 0.02, 'G_u = R_u − π_u(E_tare+E_cmp+E_com)',
        transform=ax.transAxes, fontsize=8, ha='right', va='bottom', color='gray')
plt.tight_layout()
plt.savefig('zheng_plot14_reward_case2.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved: zheng_plot14_reward_case2.png')


# ── Plot 15 — FL Accuracy per Round (main simulation) ────────────
fig, ax = plt.subplots(figsize=(10, 6))
rounds = list(range(1, len(fl_acc_rounds) + 1))
ax.plot(rounds, fl_acc_rounds, marker='o', lw=3, ms=10,
        color='royalblue', label='FL Accuracy per Round')
ax.fill_between(rounds, fl_acc_rounds, alpha=0.12, color='royalblue')
for r, a in zip(rounds, fl_acc_rounds):
    ax.annotate(f'{a:.4f}', (r, a), textcoords='offset points',
                xytext=(0, 10), ha='center', fontsize=10, fontweight='bold')
ax.set_title('FL Accuracy over Rounds — Zheng et al. (TCE 2025)',
             fontweight='bold', fontsize=13)
ax.set_xlabel('FL Round', fontsize=11)
ax.set_ylabel('Accuracy', fontsize=11)
ax.set_xticks(rounds)
ax.set_ylim(0, 1.05)
ax.tick_params(axis='x', labelsize=10)
ax.legend(fontsize=10); ax.grid(True, alpha=0.3)
ax.text(0.98, 0.02, f'FedAvg | {NUM_ROUNDS} rounds | {LOCAL_EPOCHS} local epochs',
        transform=ax.transAxes, fontsize=8, ha='right', va='bottom', color='gray')
plt.tight_layout()
plt.savefig('zheng_plot15_fl_accuracy_rounds.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved: zheng_plot15_fl_accuracy_rounds.png')


# ── Plot 16 — FL Accuracy — Case 1 (UAVs=7) ─────────────────────
fig, ax = plt.subplots(figsize=(10, 6))
x_vals = df1['n_sensors'].tolist()
y_vals = df1['acc'].tolist()
bars = ax.bar(range(len(x_vals)), y_vals, color='teal',
              edgecolor='black', linewidth=0.8, alpha=0.88, width=0.55)
for bar, v in zip(bars, y_vals):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height(),
            f'{v:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_title('FL Accuracy — Case 1 (UAVs=7)', fontweight='bold', fontsize=13)
ax.set_xlabel('Number of Sensors  [Case 1: UAVs=7]', fontsize=11)
ax.set_ylabel('Accuracy', fontsize=11)
ax.set_xticks(range(len(x_vals)))
ax.set_xticklabels([str(x) for x in x_vals], fontsize=9)
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3, axis='y')
ax.legend(['Zheng et al. (TCE 2025)'], fontsize=9)
plt.tight_layout()
plt.savefig('zheng_plot16_acc_case1.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved: zheng_plot16_acc_case1.png')


# ── Plot 17 — FL Accuracy — Case 2 (UAVs=15) ────────────────────
fig, ax = plt.subplots(figsize=(10, 6))
x_vals = df2['n_sensors'].tolist()
y_vals = df2['acc'].tolist()
bars = ax.bar(range(len(x_vals)), y_vals, color='cadetblue',
              edgecolor='black', linewidth=0.8, alpha=0.88, width=0.55)
for bar, v in zip(bars, y_vals):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height(),
            f'{v:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_title('FL Accuracy — Case 2 (UAVs=15)', fontweight='bold', fontsize=13)
ax.set_xlabel('Number of Sensors  [Case 2: UAVs=15]', fontsize=11)
ax.set_ylabel('Accuracy', fontsize=11)
ax.set_xticks(range(len(x_vals)))
ax.set_xticklabels([str(x) for x in x_vals], fontsize=9)
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3, axis='y')
ax.legend(['Zheng et al. (TCE 2025)'], fontsize=9)
plt.tight_layout()
plt.savefig('zheng_plot17_acc_case2.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved: zheng_plot17_acc_case2.png')


## Final Summary

In [ ]:
print("=" * 68)
print("  ZHENG et al. (TCE 2025) — FINAL SIMULATION SUMMARY")
print("=" * 68)
print(f"  Algorithm : Greedy Primal-Dual Auction (Algorithm 4)")
print(f"  Mapping   : CH=ACE Task | UAV+BS=Base Station")
print(f"  Farm: {AREA_SIZE}x{AREA_SIZE}m | FL rounds: {NUM_ROUNDS} | Epochs: {LOCAL_EPOCHS}")
print()
print("  Case 1 — UAVs=7, Sensors=200..1000")
print(df1[['n_sensors','K','n_uavs','matched','E_tx','E_rx',
           'T_trans','E_total','total_cost','K_reward','acc']].to_string(index=False))
print()
print("  Case 2 — UAVs=15, Sensors=200..2000 step 200")
print(df2[['n_sensors','K','n_uavs','matched','E_tx','E_rx',
           'T_trans','E_total','total_cost','K_reward','acc']].to_string(index=False))
print("=" * 68)
